# Prepare Gold Labels

This notebook creates the canonical gold-label table used throughout the thesis pipeline. Gold labels represent human-reviewed category-manager judgments and must remain separate from weak labels generated by Snorkel. The goal is to produce one clean, validated, joinable file that can be merged into the feature-engineered contract dataset before Stage 1 and Stage 2 experiments.

The notebook also creates candidate samples for future labeling. Candidate sampling is only used to select contracts for human review; it does not create synthetic gold labels.

## 1. Environment Setup

We first load standard libraries, define project paths, and configure display settings. The notebook is intended to be run from the `notebooks/` directory, so all paths are defined relative to the project root.

In [ ]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "Data" / "processed"
DATA_MANUAL = PROJECT_ROOT / "Data" / "manual"
REPORTS_DIR = PROJECT_ROOT / "reports" / "gold_labels"

DATA_MANUAL.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Manual data folder:", DATA_MANUAL)
print("Processed data folder:", DATA_PROCESSED)
print("Reports folder:", REPORTS_DIR)

## 2. Load the Contract Feature Dataset

The gold labels must be validated against the actual contract universe used in the modeling pipeline. This prevents category-manager labels from referring to contracts that were filtered out upstream or have incompatible identifiers.

In [ ]:
FEATURE_DATA_PATH = DATA_PROCESSED / "contract_with_features_labeled.csv"

if not FEATURE_DATA_PATH.exists():
    FEATURE_DATA_PATH = DATA_PROCESSED / "contracts_with_features.csv"

if not FEATURE_DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find contract feature dataset. Expected either "
        "contract_with_features_labeled.csv or contracts_with_features.csv in Data/processed/."
    )

df_features = pd.read_csv(FEATURE_DATA_PATH, low_memory=False)

print("Loaded:", FEATURE_DATA_PATH.name)
print("Shape:", df_features.shape)
print("Unique contracts:", df_features["contract_id"].nunique() if "contract_id" in df_features else "contract_id missing")
print("Departments:", df_features["department"].nunique() if "department" in df_features else "department missing")

## 3. Define or Load Manual Gold Labels

The preferred long-term workflow is to maintain manual labels in `Data/manual/gold_labels_manual.csv`. The fallback dictionary below is useful while the label set is still small and evolving. Each row should represent a human-reviewed contract-level label.

`gold_y = 1` means the contract should be considered a renegotiation candidate. `gold_y = 0` means the contract should not currently be prioritized for renegotiation.

In [ ]:
MANUAL_LABEL_PATH = DATA_MANUAL / "gold_labels_manual.csv"

# Optional fallback while collecting labels.
# Replace or extend this dictionary when new category-manager labels arrive.
# Keep contract IDs as strings to avoid integer/float formatting issues.
gold_label_records = [
    # Example format:
    # {
    #     "contract_id": "12345",
    #     "contract_number": "CN-001",
    #     "department": "Logistics",
    #     "gold_y": 1,
    #     "label_source": "category_manager",
    #     "label_date": "2026-04-14",
    #     "notes": "Clear renegotiation candidate due to renewal pressure.",
    # },
]

if MANUAL_LABEL_PATH.exists():
    df_gold_raw = pd.read_csv(MANUAL_LABEL_PATH, dtype={"contract_id": "string", "contract_number": "string"})
    print("Loaded manual labels from:", MANUAL_LABEL_PATH)
else:
    df_gold_raw = pd.DataFrame(gold_label_records)
    print("Loaded manual labels from fallback dictionary.")

if df_gold_raw.empty:
    print("WARNING: No manual gold labels loaded yet. Add labels to the dictionary or Data/manual/gold_labels_manual.csv.")
else:
    display(df_gold_raw.head())
    print("Raw gold label shape:", df_gold_raw.shape)

## 4. Standardize Gold Label Schema

This section enforces a single canonical schema. A stable schema is important because the gold labels are reused across Stage 1 validation, Stage 2 support/query construction, and thesis diagnostics.

In [ ]:
REQUIRED_GOLD_COLUMNS = [
    "contract_id",
    "contract_number",
    "department",
    "gold_y",
    "label_source",
    "label_date",
    "notes",
]

for col in REQUIRED_GOLD_COLUMNS:
    if col not in df_gold_raw.columns:
        df_gold_raw[col] = pd.NA

df_gold = df_gold_raw[REQUIRED_GOLD_COLUMNS].copy()

# Identifier normalization.
df_gold["contract_id"] = df_gold["contract_id"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()
df_gold["contract_number"] = df_gold["contract_number"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()
df_gold["department"] = df_gold["department"].astype("string").str.strip()

# Label normalization.
df_gold["gold_y"] = pd.to_numeric(df_gold["gold_y"], errors="coerce")

invalid_labels = df_gold[~df_gold["gold_y"].isin([0, 1]) & df_gold["gold_y"].notna()]
if not invalid_labels.empty:
    raise ValueError("Invalid gold_y values detected. Only 0 and 1 are allowed.")

df_gold["gold_y"] = df_gold["gold_y"].astype("Int64")

# Fill metadata defaults.
df_gold["label_source"] = df_gold["label_source"].fillna("category_manager")
df_gold["label_date"] = df_gold["label_date"].fillna(str(date.today()))
df_gold["notes"] = df_gold["notes"].fillna("")

# Remove empty rows.
df_gold = df_gold[df_gold["gold_y"].notna()].copy()

display(df_gold.head())
print("Standardized gold label shape:", df_gold.shape)

## 5. Validate Gold Labels Against the Contract Universe

A gold label is only useful if it maps to a contract that exists in the modeling dataset. This validation checks whether labeled contracts can be found by `contract_id` and, where available, by `contract_number`. Any unmatched labels should be investigated before training.

In [ ]:
df_features_ids = df_features.copy()

for col in ["contract_id", "contract_number"]:
    if col in df_features_ids.columns:
        df_features_ids[col] = df_features_ids[col].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()

feature_contract_ids = set(df_features_ids["contract_id"].dropna().unique()) if "contract_id" in df_features_ids else set()
feature_contract_numbers = set(df_features_ids["contract_number"].dropna().unique()) if "contract_number" in df_features_ids else set()

df_gold["exists_by_contract_id"] = df_gold["contract_id"].isin(feature_contract_ids)
df_gold["exists_by_contract_number"] = df_gold["contract_number"].isin(feature_contract_numbers)
df_gold["exists_in_features"] = df_gold["exists_by_contract_id"] | df_gold["exists_by_contract_number"]

unmatched = df_gold[~df_gold["exists_in_features"]].copy()

print("Gold labels:", len(df_gold))
print("Matched labels:", int(df_gold["exists_in_features"].sum()))
print("Unmatched labels:", len(unmatched))

if not unmatched.empty:
    display(unmatched)
    unmatched.to_csv(REPORTS_DIR / "unmatched_gold_labels.csv", index=False)
    print("Saved unmatched labels to reports/gold_labels/unmatched_gold_labels.csv")

## 6. Resolve Duplicate or Conflicting Labels

Gold labels should be unique at the contract level. If the same contract receives both a positive and negative label, the conflict must be resolved manually. Conflicting labels are not automatically averaged or overwritten because this would contaminate the ground truth used for evaluation.

In [ ]:
# Prefer contract_id when available; otherwise fall back to contract_number.
df_gold["gold_join_key"] = np.where(
    df_gold["contract_id"].notna() & (df_gold["contract_id"].astype(str) != ""),
    df_gold["contract_id"].astype(str),
    df_gold["contract_number"].astype(str),
)

conflict_check = (
    df_gold.groupby("gold_join_key")
    .agg(
        n_labels=("gold_y", "count"),
        n_unique_labels=("gold_y", "nunique"),
        labels=("gold_y", lambda x: sorted(x.dropna().unique().tolist())),
        departments=("department", lambda x: sorted(x.dropna().unique().tolist())),
    )
    .reset_index()
)

conflicts = conflict_check[conflict_check["n_unique_labels"] > 1].copy()

if not conflicts.empty:
    display(conflicts)
    conflicts.to_csv(REPORTS_DIR / "conflicting_gold_labels.csv", index=False)
    raise ValueError("Conflicting gold labels found. Resolve them before continuing.")

# Keep one label per contract key.
df_gold_clean = (
    df_gold.sort_values(["gold_join_key", "label_date"])
    .drop_duplicates(subset=["gold_join_key"], keep="last")
    .copy()
)

print("Gold labels after duplicate resolution:", len(df_gold_clean))

## 7. Gold Label Inventory by Department

This table is the central diagnostic for deciding whether Stage 2 is mathematically feasible. Meta-learning requires departments with both positive and negative examples. A department with only one class cannot support AUROC-based evaluation or balanced support/query episodes.

In [ ]:
df_gold_inventory = (
    df_gold_clean.groupby("department", dropna=False)
    .agg(
        gold_total=("gold_y", "count"),
        gold_yes=("gold_y", "sum"),
    )
    .reset_index()
)

df_gold_inventory["gold_no"] = df_gold_inventory["gold_total"] - df_gold_inventory["gold_yes"]
df_gold_inventory["gold_yes_rate"] = df_gold_inventory["gold_yes"] / df_gold_inventory["gold_total"]

df_gold_inventory = df_gold_inventory.sort_values("gold_total", ascending=False)

display(df_gold_inventory)
df_gold_inventory.to_csv(REPORTS_DIR / "gold_label_inventory_by_department.csv", index=False)

## 8. Stage 2 Readiness Check

For k-shot meta-learning, each department needs enough positive and negative labels to form a support set and still leave query examples for evaluation. The rule used here is conservative: a department is valid for k-shot if it has at least `k + 1` positive and `k + 1` negative labels.

In [ ]:
K_VALUES = [2, 5, 10]
TARGET_DEPARTMENT = "Logistics"

df_stage2_readiness = df_gold_inventory.copy()

for k in K_VALUES:
    df_stage2_readiness[f"valid_k{k}"] = (
        (df_stage2_readiness["gold_yes"] >= k + 1)
        & (df_stage2_readiness["gold_no"] >= k + 1)
    )

def suggest_role(row):
    if row["department"] == TARGET_DEPARTMENT:
        return "target_candidate"
    if row["gold_total"] == 0:
        return "unlabeled"
    if row["gold_yes"] == 0 or row["gold_no"] == 0:
        return "invalid_one_class"
    if row.get("valid_k2", False):
        return "source_task"
    return "invalid_too_few"

df_stage2_readiness["suggested_role"] = df_stage2_readiness.apply(suggest_role, axis=1)

display(df_stage2_readiness)
df_stage2_readiness.to_csv(REPORTS_DIR / "stage2_gold_label_readiness.csv", index=False)

## 9. Save the Canonical Gold Label Table

The output of this section is the authoritative gold-label file used by downstream notebooks. The file should be version controlled if it does not contain confidential information, or stored securely with a clear versioning procedure if it does.

In [ ]:
GOLD_CLEAN_PATH = DATA_PROCESSED / "gold_labels_clean.csv"

df_gold_clean_export = df_gold_clean[
    [
        "contract_id",
        "contract_number",
        "department",
        "gold_y",
        "label_source",
        "label_date",
        "notes",
        "gold_join_key",
    ]
].copy()

df_gold_clean_export.to_csv(GOLD_CLEAN_PATH, index=False)

print("Saved clean gold labels to:", GOLD_CLEAN_PATH)
print("Shape:", df_gold_clean_export.shape)

## 10. Join Gold Labels onto the Feature Dataset

This join produces a modeling-ready dataset with gold labels attached. Snorkel weak labels and gold labels serve different purposes: Snorkel labels provide scalable training signal, while gold labels are used for validation, support/query construction, and final evaluation.

In [ ]:
df_features_joined = df_features_ids.copy()

# Remove old gold columns if present to avoid stale labels.
for col in ["gold_y", "gold_department", "label_source", "label_date", "notes"]:
    if col in df_features_joined.columns:
        df_features_joined = df_features_joined.drop(columns=[col])

join_cols = ["contract_id", "contract_number", "department", "gold_y", "label_source", "label_date", "notes"]
df_gold_for_join = df_gold_clean_export[join_cols].copy()

# Primary join by contract_id.
df_features_with_gold = df_features_joined.merge(
    df_gold_for_join.drop(columns=["contract_number"], errors="ignore"),
    on="contract_id",
    how="left",
    suffixes=("", "_gold"),
)

# Preserve department from the gold label as a separate audit column.
if "department_gold" in df_features_with_gold.columns:
    df_features_with_gold = df_features_with_gold.rename(columns={"department_gold": "gold_department"})
else:
    df_features_with_gold["gold_department"] = pd.NA

print("Joined feature table shape:", df_features_with_gold.shape)
print("Gold-labeled rows after join:", int(df_features_with_gold["gold_y"].notna().sum()))
print("Gold-labeled unique contracts after join:", df_features_with_gold.loc[df_features_with_gold["gold_y"].notna(), "contract_id"].nunique())

## 11. Save Feature Dataset with Gold Labels

This saved dataset can be used for Stage 1 gold-label validation and Stage 2 support/query construction. If the dataset already contains Snorkel outputs, those columns are preserved.

In [ ]:
FEATURES_WITH_GOLD_PATH = DATA_PROCESSED / "contract_with_features_gold_joined.csv"

df_features_with_gold.to_csv(FEATURES_WITH_GOLD_PATH, index=False)

print("Saved feature dataset with gold labels to:", FEATURES_WITH_GOLD_PATH)
print("Shape:", df_features_with_gold.shape)

# Candidate Sampling for Additional Gold Labels

The next cells create candidate lists for category managers. These samples are not gold labels. They are only a structured way to request more human annotations, especially from departments/classes that are currently underrepresented.

## 12. Candidate Sampling Strategy

For each selected department, we sample three groups of contracts:

1. High-probability candidates: likely renegotiation examples.
2. Low-probability candidates: likely non-renegotiation examples.
3. Uncertain candidates: useful for clarifying ambiguous decision boundaries.

This helps collect balanced labels efficiently without pretending that model scores are ground truth.

In [ ]:
CANDIDATE_DEPARTMENTS = [
    "Logistics",
    "Packaging Material",
    "Drug Product Outsourcing",
    "Drug Substance Outsourcing",
    "Raw Materials & Energy",
    "Quality, Production Services & Supplies",
]

N_HIGH = 10
N_LOW = 10
N_UNCERTAIN = 10
SCORE_COL = "renegotiation_prob"

if SCORE_COL not in df_features_with_gold.columns:
    raise ValueError(f"Cannot sample candidates because {SCORE_COL} is missing.")

candidate_base = df_features_with_gold.copy()

# One row per contract for review candidates.
candidate_base = (
    candidate_base.sort_values(["contract_id", "observation_year"] if "observation_year" in candidate_base else ["contract_id"])
    .drop_duplicates(subset=["contract_id"], keep="last")
    .copy()
)

# Exclude already labeled contracts.
candidate_base = candidate_base[candidate_base["gold_y"].isna()].copy()

print("Candidate pool unique contracts:", candidate_base["contract_id"].nunique())

In [ ]:
def sample_label_candidates(
    df: pd.DataFrame,
    department: str,
    score_col: str = "renegotiation_prob",
    n_high: int = 10,
    n_low: int = 10,
    n_uncertain: int = 10,
) -> pd.DataFrame:
    dept_df = df[df["department"] == department].copy()
    dept_df = dept_df[dept_df[score_col].notna()].copy()

    if dept_df.empty:
        return pd.DataFrame()

    high = dept_df.sort_values(score_col, ascending=False).head(n_high).copy()
    high["candidate_bucket"] = "likely_yes_high_score"

    low = dept_df.sort_values(score_col, ascending=True).head(n_low).copy()
    low["candidate_bucket"] = "likely_no_low_score"

    uncertain = dept_df.assign(distance_to_half=(dept_df[score_col] - 0.5).abs())
    uncertain = uncertain.sort_values("distance_to_half", ascending=True).head(n_uncertain).copy()
    uncertain["candidate_bucket"] = "uncertain_near_0_5"

    out = pd.concat([high, low, uncertain], ignore_index=True)
    out = out.drop_duplicates(subset=["contract_id"], keep="first")
    out["target_department"] = department

    keep_cols = [
        "target_department",
        "candidate_bucket",
        "contract_id",
        "contract_number",
        "contract_name",
        "supplier_id",
        "supplier_display_name",
        "department",
        score_col,
        "contract_age_years",
        "years_to_expiry_capped",
        "payment_terms",
        "incoterms",
        "contract_value_dkk",
    ]
    keep_cols = [col for col in keep_cols if col in out.columns]

    return out[keep_cols].copy()

In [ ]:
candidate_frames = []

for dept in CANDIDATE_DEPARTMENTS:
    df_candidates_dept = sample_label_candidates(
        candidate_base,
        department=dept,
        score_col=SCORE_COL,
        n_high=N_HIGH,
        n_low=N_LOW,
        n_uncertain=N_UNCERTAIN,
    )

    if not df_candidates_dept.empty:
        safe_dept_name = dept.lower().replace(" ", "_").replace("&", "and").replace("/", "_")
        output_path = REPORTS_DIR / f"gold_label_candidates_{safe_dept_name}.csv"
        df_candidates_dept.to_csv(output_path, index=False)
        candidate_frames.append(df_candidates_dept)
        print(f"Saved {len(df_candidates_dept)} candidates for {dept}: {output_path.name}")
    else:
        print(f"No candidates found for {dept}")

if candidate_frames:
    df_all_candidates = pd.concat(candidate_frames, ignore_index=True)
    df_all_candidates.to_csv(REPORTS_DIR / "gold_label_candidates_all_departments.csv", index=False)
    display(df_all_candidates.head(30))
else:
    print("No candidate files created.")

## 13. Final Summary

The final summary connects the label inventory to the thesis methodology. If the gold-label distribution is balanced across several departments, Stage 2 meta-learning becomes more defensible. If labels remain concentrated in one department or one class, the thesis should frame Stage 2 as constrained few-shot adaptation rather than fully general meta-learning.

In [ ]:
total_gold = len(df_gold_clean_export)
total_yes = int(df_gold_clean_export["gold_y"].sum()) if total_gold else 0
total_no = int(total_gold - total_yes)

print("--- GOLD LABEL SUMMARY ---")
print("Total gold labels:", total_gold)
print("Gold yes:", total_yes)
print("Gold no:", total_no)
print("Departments with gold labels:", df_gold_clean_export["department"].nunique() if total_gold else 0)

for k in K_VALUES:
    valid_sources = df_stage2_readiness[
        (df_stage2_readiness[f"valid_k{k}"])
        & (df_stage2_readiness["department"] != TARGET_DEPARTMENT)
    ]
    print(f"Valid source departments for k={k}:", len(valid_sources))

logistics_row = df_stage2_readiness[df_stage2_readiness["department"] == TARGET_DEPARTMENT]
if not logistics_row.empty:
    display(logistics_row)
else:
    print("No Logistics gold labels found yet.")